# Tutorial: Palette catalog summaries

Audience:
- Developers who want to group a CSV palette catalog by primary color using only the Python standard library.

Prerequisites:
- Basic Python (functions, dictionaries, lists).
- Familiarity with reading CSV files.

Learning goals:
- By the end, the reader can write a function that reads a CSV catalog, normalizes key fields, groups rows, and sorts the groups deterministically using only the standard library.


## Outline

1. Setup
2. Define `summarize_palettes(csv_path)`
3. Run it on the catalog
4. Inspect the results
5. Exercises


In [ ]:
# Setup cell: keep it short and deterministic
from __future__ import annotations

import csv
from collections import defaultdict

CATALOG_PATH = "/inputs/colors.csv"
CATALOG_PATH


## Step 1 - Define the grouping function

`summarize_palettes(csv_path)` reads the CSV using its header row, strips
surrounding whitespace from the `Primary (Hex)` and `Product Type` fields,
uppercases the primary color to normalize it, and groups rows by that
normalized primary color.

Each group becomes an object with:
- `primary`: the normalized (stripped, uppercased) primary color text.
- `count`: the number of rows in the group, including duplicate rows.
- `product_types`: the sorted, unique, stripped product-type texts in the group.

Groups are ordered by `count` descending, then by `primary` ascending
(plain Python lexicographic string sorting).


In [ ]:
def summarize_palettes(csv_path: str) -> list[dict]:
    groups: dict[str, dict] = {}

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            raw_primary = row.get("Primary (Hex)", "")
            raw_product_type = row.get("Product Type", "")

            primary = (raw_primary or "").strip().upper()
            product_type = (raw_product_type or "").strip()

            if primary not in groups:
                groups[primary] = {"count": 0, "product_types": set()}

            groups[primary]["count"] += 1
            if product_type:
                groups[primary]["product_types"].add(product_type)

    summaries = []
    for primary, data in groups.items():
        summaries.append(
            {
                "primary": primary,
                "count": data["count"],
                "product_types": sorted(data["product_types"]),
            }
        )

    summaries.sort(key=lambda item: (-item["count"], item["primary"]))
    return summaries


summarize_palettes


## Step 2 - Run it on the real catalog

Call the function on the bundled catalog and inspect how many distinct
normalized primary colors it found.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')
len(palette_summary)


## Step 3 - Inspect a few groups

Look at the first few groups, ordered by count descending then primary
ascending, to sanity-check the grouping and sorting.


In [ ]:
palette_summary[:3]


## Exercises

- Modify the function to also group by `Product Type` as a secondary key and
  see how the groups change.
- Create a tiny in-memory CSV string with duplicate rows and lowercase hex
  values, write it to a temp file, and confirm duplicates are counted and
  case is normalized.
- Pass a header-only CSV (no data rows) and confirm the function returns an
  empty list.
- Try a catalog with quoted product-type values containing commas (for
  example, `"Design, System"`) and confirm the value survives intact after
  stripping.
